# Conjugate gradient methods

Three variants ship in MOpt:

- `QuadraticCG` — linear CG, exact for a quadratic in at most $n$ steps, no line search
- `ConjugateGradient` — nonlinear CG with a pluggable $\beta$ rule and a Wolfe line search
- `ArmijoModifiedCG` — nonlinear CG using an Armijo-type step rule instead

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

from mopt.nonlinear import (NLPProblem, QuadraticCG, ConjugateGradient,
                            ArmijoModifiedCG, fletcher_reeves, polak_ribiere)

## Linear CG on a quadratic

For $f(x) = \tfrac12 x^T A x - c^T x$ the step length is available in closed
form, so `QuadraticCG` needs no line search and terminates within $n$
iterations in exact arithmetic. It reads $A$ from `problem.hess`, which it
evaluates once and assumes constant.

In [2]:
rng = np.random.default_rng(0)
n = 5
L = np.tril(rng.uniform(-1.0, 1.0, size=(n, n)))
np.fill_diagonal(L, rng.uniform(1.0, 2.0, size=n))
A = L @ L.T                       # symmetric positive definite
c_vec = rng.normal(size=n)

quad = NLPProblem(
    f=lambda x: float(0.5 * x @ A @ x - c_vec @ x),
    x0=np.zeros(n),
    grad=lambda x: A @ x - c_vec,
    hess=lambda x: A,
)

result = QuadraticCG().solve(quad)
exact = np.linalg.solve(A, c_vec)

print(f"iters       = {result.n_iter}  (dimension n = {n})")
print(f"|x - exact| = {np.linalg.norm(result.x - exact):.3e}")
print(f"cond(A)     = {np.linalg.cond(A):.1f}")

iters       = 5  (dimension n = 5)
|x - exact| = 2.523e-16
cond(A)     = 10.4


`QuadraticCG` is only valid for a quadratic — it does not check, and will
happily return a wrong answer otherwise. Use `ConjugateGradient` for anything
else.

## Nonlinear CG on a quartic

Here the $\beta$ rule is what carries curvature information between
iterations. Fletcher-Reeves and Polak-Ribiere are both provided and are
swapped through the `beta=` argument.

In [3]:
Q = np.array([
    [1.42, 0.06, 0.09, 0.09, 0.19],
    [0.06, 1.57, -0.24, 0.13, 0.23],
    [0.09, -0.24, 1.26, 0.11, -0.13],
    [0.09, 0.13, 0.11, 1.96, 0.02],
    [0.19, 0.23, -0.13, 0.02, 1.24],
])
M = np.array([
    [7.16, 0.26, 0.30, 1.01, 0.06],
    [0.26, 6.54, -0.44, -0.71, -0.04],
    [0.30, -0.44, 5.87, -0.44, -0.24],
    [1.01, -0.71, -0.44, 6.06, 2.48],
    [0.06, -0.04, -0.24, 2.48, 4.04],
])
b = np.array([1.7, -0.7, -1.9, 0.8, -2.0])
c = np.array([-1.1, 0.3, -0.9, 0.4, -1.4])

f = lambda x: float((x @ Q @ x + b @ x) ** 2 + x @ M @ x + c @ x)
grad_f = lambda x: (4.0 * (Q @ x) + 2.0 * b) * (x @ Q @ x + b @ x) + 2.0 * (M @ x) + c

In [4]:
rng_q = np.random.default_rng(42)
inits = np.vstack([np.zeros(5), rng_q.uniform(-2.0, 2.0, size=(5, 5))])

solvers = [
    ("CG (Fletcher-Reeves)", ConjugateGradient(beta=fletcher_reeves, max_iter=5000)),
    ("CG (Polak-Ribiere)",   ConjugateGradient(beta=polak_ribiere, max_iter=5000)),
    ("ArmijoModifiedCG",     ArmijoModifiedCG(max_iter=5000)),
]

rows = []
for x0 in inits:
    problem = NLPProblem(f=f, x0=x0, grad=grad_f)
    x_ref = minimize(f, x0, jac=grad_f, method="BFGS").x
    for name, solver in solvers:
        result = solver.solve(problem)
        if result.success:
            # only a converged run is meant to match the reference
            np.testing.assert_allclose(result.x, x_ref, atol=1e-4)
        rows.append({
            "x0": tuple(np.round(x0, 3)),
            "method": name,
            "ok": result.success,
            "iters": result.n_iter,
            "f": result.fun,
            "|x - x_scipy|": np.linalg.norm(result.x - x_ref),
        })

table = (pd.DataFrame(rows)
         .pivot(index="x0", columns="method")
         .swaplevel(axis=1)
         .reindex(columns=pd.MultiIndex.from_product(
             [[name for name, _ in solvers], ["ok", "iters", "f", "|x - x_scipy|"]])))
table

CG (Fletcher-Reeves)                  \
                                                        ok iters         f   
x0                                                                           
(-1.091, 0.218, -1.745, 1.311, 0.527)                 True    33 -0.245147   
(-0.517, 1.707, 0.575, 1.291, -0.226)                 True    21 -0.245147   
(0.0, 0.0, 0.0, 0.0, 0.0)                             True    19 -0.245147   
(1.032, -0.582, 1.883, 1.572, 1.114)                  True    21 -0.245147   
(1.096, -0.244, 1.434, 0.789, -1.623)                 True    19 -0.245147   
(1.902, 1.045, 1.144, -1.488, -0.198)                 True    28 -0.245147   

                                                    CG (Polak-Ribiere)        \
                                      |x - x_scipy|                 ok iters   
x0                                                                             
(-1.091, 0.218, -1.745, 1.311, 0.527)  7.339946e-07               True    25   
(-0.517, 1.707, 0.575, 1.291, -0.226)  3.870102e-07               True    25   
(0.0, 0.0, 0.0, 0.0, 0.0)              1.251972e-07               True    21   
(1.032, -0.582, 1.883, 1.572, 1.114)   4.576386e-07               True    29   
(1.096, -0.244, 1.434, 0.789, -1.623)  3.138348e-07               True    26   
(1.902, 1.045, 1.144, -1.488, -0.198)  1.535803e-06              False     1   

                                                                \
                                               f |x - x_scipy|   
x0                                                               
(-1.091, 0.218, -1.745, 1.311, 0.527)  -0.245147  8.854245e-07   
(-0.517, 1.707, 0.575, 1.291, -0.226)  -0.245147  4.529990e-07   
(0.0, 0.0, 0.0, 0.0, 0.0)              -0.245147  1.227483e-07   
(1.032, -0.582, 1.883, 1.572, 1.114)   -0.245147  4.518267e-07   
(1.096, -0.244, 1.434, 0.789, -1.623)  -0.245147  3.298079e-07   
(1.902, 1.045, 1.144, -1.488, -0.198)  29.791986  2.237841e+00   

                                      ArmijoModifiedCG                  \
                                                    ok iters         f   
x0                                                                       
(-1.091, 0.218, -1.745, 1.311, 0.527)             True    40 -0.245147   
(-0.517, 1.707, 0.575, 1.291, -0.226)             True    50 -0.245147   
(0.0, 0.0, 0.0, 0.0, 0.0)                         True    39 -0.245147   
(1.032, -0.582, 1.883, 1.572, 1.114)              True    35 -0.245147   
(1.096, -0.244, 1.434, 0.789, -1.623)             True    46 -0.245147   
(1.902, 1.045, 1.144, -1.488, -0.198)             True    56 -0.245147   

                                                     
                                      |x - x_scipy|  
x0                                                   
(-1.091, 0.218, -1.745, 1.311, 0.527)  6.739216e-07  
(-0.517, 1.707, 0.575, 1.291, -0.226)  4.340264e-07  
(0.0, 0.0, 0.0, 0.0, 0.0)              1.399431e-07  
(1.032, -0.582, 1.883, 1.572, 1.114)   5.000428e-07  
(1.096, -0.244, 1.434, 0.789, -1.623)  5.087803e-07  
(1.902, 1.045, 1.144, -1.488, -0.198)  1.429708e-06

## Polak-Ribiere can fail to produce a descent direction

Fletcher-Reeves paired with a strong Wolfe line search is guaranteed to keep
generating descent directions. Polak-Ribiere is not — and on this problem it
genuinely stops on one of the six starts. MOpt detects that and says so rather
than stepping uphill.

In [5]:
failed = [r for r in rows if not r["ok"]]
print(f"{len(failed)} run(s) did not converge\n")

for r in failed:
    problem = NLPProblem(f=f, x0=np.array(r["x0"]), grad=grad_f)
    message = ConjugateGradient(beta=polak_ribiere, max_iter=5000).solve(problem).message
    print(f"x0 = {r['x0']}\n  {message}")

1 run(s) did not converge

x0 = (np.float64(1.902), np.float64(1.045), np.float64(1.144), np.float64(-1.488), np.float64(-0.198))
  Direction is not a descent direction at iteration 1; this beta rule and line search do not guarantee descent.


`ArmijoModifiedCG` uses the same Polak-Ribiere $\beta$ but a different step
rule, and comes through on every start — the pairing of $\beta$ with the step
rule is what matters, not $\beta$ alone.

In [6]:
summary = (pd.DataFrame(rows)
           .groupby("method")
           .agg(converged=("ok", "sum"), runs=("ok", "size"),
                mean_iters=("iters", "mean")))
summary["mean_iters"] = summary["mean_iters"].round(1)
print(summary.to_string())

                      converged  runs  mean_iters
method                                           
ArmijoModifiedCG              6     6        44.3
CG (Fletcher-Reeves)          6     6        23.5
CG (Polak-Ribiere)            5     6        21.2
